In [7]:
!pip install thefuzz

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from itertools import combinations, permutations
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully.")

All libraries imported successfully.


In [9]:

FILE_PATH = "Food_Inspections_20240215.csv"  # Update path if needed

df = pd.read_csv(FILE_PATH)

# Standardize column names (strip whitespace)
df.columns = df.columns.str.strip()

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\n Columns ({df.shape[1]}):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2}. {col}")

Dataset loaded: 267,531 rows × 17 columns

 Columns (17):
   1. Inspection ID
   2. DBA Name
   3. AKA Name
   4. License #
   5. Facility Type
   6. Risk
   7. Address
   8. City
   9. State
  10. Zip
  11. Inspection Date
  12. Inspection Type
  13. Results
  14. Violations
  15. Latitude
  16. Longitude
  17. Location


In [10]:
import re
import time
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from thefuzz import fuzz

# ══════════════════════════════════════════════════════════════
#  CONSTANTS
# ══════════════════════════════════════════════════════════════

# ── Geographic ────────────────────────────────────────────────
CHICAGO_LAT_MIN =  41.60
CHICAGO_LAT_MAX =  42.05
CHICAGO_LON_MIN = -88.00
CHICAGO_LON_MAX = -87.20

TARGET_STATE          = "IL"
TARGET_CITY           = "CHICAGO"
TARGET_ZIP_PREFIXES   = ("606", "607", "608")
TARGET_ZIP_LENGTH     = 5
CITY_FUZZ_THRESHOLD   = 80
DROP_NON_TARGET_CITIES = True

CITY_MANUAL_MAP = {
    "INACTIVE": "CHICAGO",
    "NAN":      "CHICAGO",
}

# ── Risk ──────────────────────────────────────────────────────
TARGET_RISKS = ["RISK 1 (HIGH)", "RISK 2 (MEDIUM)", "RISK 3 (LOW)"]

# ── Facility Type ─────────────────────────────────────────────
TYPE_DESCRIPTIONS = {
    "Bakery": "Bakery bread pastry cakes baked goods patisserie",
    "Banquet Hall": "Banquet hall event venue catering reception wedding party room",
    "Candy Store": "Candy store sweets confectionery sugar snacks chocolate gelato",
    "Caterer": "Caterer catering food service commissary kitchen prep cooking",
    "Coffee Shop": "Coffee shop cafe espresso drinks tea kiosk smoothie beverages",
    "Day Care Center (for ages less than 2)": "Day care infant toddler baby nursery under 2 years children",
    "Day Care Center (for ages 2 - 6)": "Day care preschool children ages 2 to 6 years kindergarten",
    "Day Care Center (combo, for ages less than 2 and 2 - 6 combined)": "Day care combo infant toddler preschool all ages children facility",
    "Gas Station": "Gas station fuel petrol convenience store mini mart service station",
    "Golden Diner": "Golden diner classic American diner breakfast lunch",
    "Grocery Store": "Grocery store supermarket food retail produce market butcher",
    "Hospital": "Hospital medical healthcare clinic nursing patient care",
    "Long Term Care Center (Nursing Home)": "Nursing home long term care assisted living senior elderly residential care",
    "Liquor Store": "Liquor store alcohol beer wine spirits packaged goods",
    "Mobile Food Dispenser": "Mobile food truck cart vendor pushcart dispenser frozen dessert street food",
    "Restaurant": "Restaurant dining food service eating establishment bar grill tavern pub",
    "Paleteria": "Paleteria ice cream paleta Mexican frozen treats sorbet",
    "School": "School education classroom students learning academy cafeteria charter",
    "Shelter": "Shelter homeless housing refuge non-profit aid charity social services",
    "Tavern": "Tavern bar pub lounge brewery brewpub alcohol drinks nightclub hookah",
    "Social Club": "Social club VFW hall community organization members recreation",
    "Wholesaler": "Wholesaler wholesale distributor bulk food warehouse distribution",
    "Wrigley Field Rooftop": "Wrigley field rooftop Chicago Cubs baseball stadium venue",
}

ABBREV_MAP = {
    r"\bREST\b": "RESTAURANT",        r"\bGROC\b": "GROCERY",
    r"\bLIQ\b": "LIQUOR",             r"\bMFD\b": "MOBILE FOOD DISPENSER",
    r"\bBAK\b": "BAKERY",             r"\bCAT\b": "CATERER",
    r"\bBKRY\b": "BAKERY",            r"\bDELI\b": "DELICATESSEN",
    r"\bMOBIL\b": "MOBILE",           r"\bRSTAURANT\b": "RESTAURANT",
    r"\bRESATURANT\b": "RESTAURANT",  r"\bTENT\b": "TEMPORARY",
    r"\bCOMMISARY\b": "COMMISSARY",   r"\bCOMMIASARY\b": "COMMISSARY",
    r"\bPREPACKAGE\b": "PREPACKAGED", r"\bDAYCARE\b": "DAY CARE",
    r"\bNP\b": "NON PROFIT",          r"\bVFW\b": "VETERANS SOCIAL CLUB",
    r"\bGYM\b": "FITNESS CENTER",     r"\bSPA\b": "WELLNESS CENTER",
    r"\bPUB\b": "TAVERN BAR",
}

MANUAL_OVERRIDES = {
    # "exact original Facility Type string": "Official Type",
}

CHICAGO_ADDRESS_PATTERN = re.compile(r"""
    ^
    \d+(?:-\d+)?(?:\s+\d+/\d+)?
    (?:\s+[NSEW](?:ORTH|OUTH|AST|EST)?)?
    (?:\s+[A-Z0-9\'.]+)+
    \s+(?:ST|AVE|BLVD|DR|RD|CT|LN|PL|PKWY|HWY|CIR|TER|TRL|WAY|
            MARKET|RUN|ROW|PATH|WALK|LOOP|XING|PASS|SQ|EXPY|FRWY)
    (?:\s+(?:APT|STE|UNIT|FL|FLOOR|BLDG|RM|\#|LBBY)[\s\w\#-]*)?
    $
""", re.VERBOSE)




# ══════════════════════════════════════════════════════════════
#  STEP 0 — Violations
#  Normalises delimiter + code prefixes for unknown formats (no drops)
# ══════════════════════════════════════════════════════════════

def clean_violations(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    if "Violations" not in result.columns:
        return result

    viol = result["Violations"].copy()
    sample = viol.dropna().head(500)
    if sample.str.contains(r"\|\|", regex=True).mean() > 0.3:
        delim = "||"
    elif sample.str.contains(r"\|", regex=True).mean() > 0.3:
        delim = "|"
    elif sample.str.contains(";", regex=False).mean() > 0.3:
        delim = ";"
    elif sample.str.contains(r"\n", regex=True).mean() > 0.3:
        delim = "\n"
    else:
        delim = "|"

    if delim != "|":
        viol = viol.str.replace(delim, "|", regex=False)

    viol = viol.str.replace(r"\s*\|\s*", " | ", regex=True)
    viol = viol.str.replace(r"(\|\s*)+\|", "|", regex=True)
    viol = viol.str.strip("| ")
    viol = viol.str.replace(r"(?i)(?:violation|code)[:\s]*#?\s*(\d+)",
                            r"\1", regex=True)
    viol = viol.str.replace(r"#(\d+)", r"\1", regex=True)

    result["Violations"] = viol
    n_non_null = viol.notna().sum()
    print(f"  Violations normalised : {n_non_null:,} non-null entries")
    print(f"  Delimiter detected    : {repr(delim)}")
    return result

# ══════════════════════════════════════════════════════════════
#  STEP 1 — Risk
#  Drops rows whose Risk value is not in TARGET_RISKS
# ══════════════════════════════════════════════════════════════

def clean_risk(df: pd.DataFrame, col="Risk") -> pd.DataFrame:
    result = df.copy()
    result[col] = result[col].astype(str).str.upper().str.strip()

    invalid_mask = ~result[col].isin(TARGET_RISKS)
    print(f"  Valid Risk rows   : {(~invalid_mask).sum():,}")
    print(f"  Dropped (invalid) : {invalid_mask.sum():,}")
    if invalid_mask.any():
        print(f"  Invalid values seen:")
        print(result.loc[invalid_mask, col].value_counts(dropna=False).to_string())

    return result[~invalid_mask].copy()


# ══════════════════════════════════════════════════════════════
#  STEP 2 — Address
#  Cleans and flags addresses — never drops rows
# ══════════════════════════════════════════════════════════════

def clean_address(address: str) -> str:
    if not isinstance(address, str):
        return ""
    return " ".join(address.upper().split())

def is_valid_chicago_address(address: str) -> bool:
    return bool(CHICAGO_ADDRESS_PATTERN.match(address))

def clean_address_column(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    result["Address"] = result["Address"].apply(clean_address)
    result["address_is_valid"] = result["Address"].apply(is_valid_chicago_address)

    n_invalid = (~result["address_is_valid"]).sum()
    print(f"  Valid addresses   : {result['address_is_valid'].sum():,}")
    print(f"  Invalid addresses : {n_invalid:,} (flagged, not dropped)")
    return result


# ══════════════════════════════════════════════════════════════
#  STEP 3 — Coordinates → State → City → Zip
#  3a: Fix State/City from Location coords (no drops)
#  3b: Clean State  → drop non-IL
#  3c: Clean City   → fuzz match → drop non-Chicago
#  3d: Clean Zip    → drop invalid zip codes
# ══════════════════════════════════════════════════════════════

def parse_coord(value):
    if pd.isna(value):
        return None
    match = re.search(r"-?\d+\.\d+", str(value).strip())
    return float(match.group()) if match else None

def parse_location_pair(value):
    if pd.isna(value):
        return None, None
    nums = re.findall(r"-?\d+\.\d+", str(value).strip())
    return (float(nums[0]), float(nums[1])) if len(nums) >= 2 else (None, None)

def is_inside_chicago(lat, lon):
    if lat is None or lon is None:
        return False
    return (CHICAGO_LAT_MIN <= lat <= CHICAGO_LAT_MAX and
            CHICAGO_LON_MIN <= lon <= CHICAGO_LON_MAX)

def build_location(lat, lon):
    return f"({lat}, {lon})" if lat is not None and lon is not None else None

def clean_location_state_city_zip(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()

    # ── 3a: Fix State/City from coordinates ───────────────────
    print("\n  [3a] Fixing State/City from Location coordinates...")
    result["_loc_lat"] = result["Location"].apply(lambda x: parse_location_pair(x)[0])
    result["_loc_lon"] = result["Location"].apply(lambda x: parse_location_pair(x)[1])

    location_present = result["_loc_lat"].notna() & result["_loc_lon"].notna()
    inside_chicago   = result.apply(
        lambda r: is_inside_chicago(r["_loc_lat"], r["_loc_lon"]), axis=1)
    fix_mask = location_present & inside_chicago

    state_needs_fix = fix_mask & (
        result["State"].isna() |
        (result["State"].astype(str).str.strip().str.upper() != "IL"))
    city_needs_fix = fix_mask & (
        result["City"].isna() |
        (result["City"].astype(str).str.strip().str.upper() != "CHICAGO"))

    result.loc[fix_mask, "State"] = "IL"
    result.loc[fix_mask, "City"]  = "CHICAGO"
    print(f"  State corrected to IL      : {state_needs_fix.sum():,}")
    print(f"  City corrected to CHICAGO  : {city_needs_fix.sum():,}")

    # Fill null Locations with IL average
    result["_lat_f"] = result["Latitude"].apply(parse_coord)
    result["_lon_f"] = result["Longitude"].apply(parse_coord)
    valid_coords = result["_lat_f"].notna() & result["_lon_f"].notna()
    il_avg_lat   = round(result.loc[valid_coords, "_lat_f"].mean(), 6)
    il_avg_lon   = round(result.loc[valid_coords, "_lon_f"].mean(), 6)
    loc_null_mask = (
        result["Location"].isna() |
        result["Location"].astype(str).str.strip().isin(["", "nan"]))
    result.loc[loc_null_mask, "Location"]  = build_location(il_avg_lat, il_avg_lon)
    result.loc[loc_null_mask, "Latitude"]  = il_avg_lat
    result.loc[loc_null_mask, "Longitude"] = il_avg_lon
    print(f"  Null Locations filled      : {loc_null_mask.sum():,}  "
          f"(avg: {il_avg_lat}, {il_avg_lon})")

    result.drop(columns=["_loc_lat", "_loc_lon", "_lat_f", "_lon_f"],
                inplace=True, errors="ignore")

    # ── 3b: State — drop non-IL ───────────────────────────────
    print("\n  [3b] Dropping non-IL rows...")
    result["State"] = result["State"].astype(str).str.upper().str.strip()
    non_il = result["State"] != TARGET_STATE
    print(f"  Dropped (non-IL) : {non_il.sum():,}")
    result = result[~non_il].copy()

    # ── 3c: City — manual map + fuzz + optional drop ──────────
    print("\n  [3c] Cleaning City...")
    result["City"] = result["City"].astype(str).str.upper().str.strip()
    result["City"] = result["City"].replace(CITY_MANUAL_MAP)

    def apply_fuzz(city_val):
        if city_val == TARGET_CITY:
            return city_val
        return TARGET_CITY if fuzz.ratio(city_val, TARGET_CITY) >= CITY_FUZZ_THRESHOLD else city_val

    result["City"] = result["City"].apply(apply_fuzz)

    non_chicago = result["City"] != TARGET_CITY
    print(f"  Non-Chicago rows : {non_chicago.sum():,}  "
          f"({'dropping' if DROP_NON_TARGET_CITIES else 'keeping'})")
    if DROP_NON_TARGET_CITIES:
        result = result[~non_chicago].copy()

    # ── 3d: Zip ───────────────────────────────────────────────
    print("\n  [3d] Cleaning Zip...")
    result["Zip"] = (result["Zip"].astype(str)
                     .str.replace(r"\.0$", "", regex=True)
                     .str.upper().str.strip())
    invalid_zip = ~(
        result["Zip"].str.startswith(TARGET_ZIP_PREFIXES, na=False) &
        (result["Zip"].str.len() == TARGET_ZIP_LENGTH))
    print(f"  Dropped (invalid Zip) : {invalid_zip.sum():,}")
    result = result[~invalid_zip].copy()

    return result.reset_index(drop=True)


# ══════════════════════════════════════════════════════════════
#  STEP 4 — Facility Type NLP
#  Never drops rows
# ══════════════════════════════════════════════════════════════

def preprocess_facility_type(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text).upper().strip()
    for pattern, replacement in ABBREV_MAP.items():
        text = re.sub(pattern, replacement, text)
    text = re.sub(r"[/\-]", " ", text)
    text = re.sub(r"[^A-Z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def clean_facility_type_column(df: pd.DataFrame, model: SentenceTransformer) -> pd.DataFrame:
    result = df.copy()

    official_labels     = list(TYPE_DESCRIPTIONS.keys())
    official_embeddings = model.encode(
        list(TYPE_DESCRIPTIONS.values()), show_progress_bar=False)

    unique_vals   = result["Facility Type"].dropna().unique().tolist()
    preprocessed  = [preprocess_facility_type(v) for v in unique_vals]
    raw_embeddings = model.encode(preprocessed, show_progress_bar=True, batch_size=64)
    sim_matrix    = cosine_similarity(raw_embeddings, official_embeddings)

    lookup = {}
    for i, raw_val in enumerate(unique_vals):
        score      = float(sim_matrix[i].max())
        matched    = official_labels[sim_matrix[i].argmax()]
        confidence = "high" if score >= 0.75 else "medium" if score >= 0.50 else "low"
        lookup[raw_val] = {
            "matched":    MANUAL_OVERRIDES.get(raw_val, matched),
            "score":      round(score, 4),
            "confidence": confidence,
        }

    result["Facility Type Original"]   = result["Facility Type"]
    result["Facility Type"]            = result["Facility Type Original"].map(
        lambda x: lookup.get(x, {}).get("matched", x))
    result["Facility Type Score"]      = result["Facility Type Original"].map(
        lambda x: lookup.get(x, {}).get("score", 0.0))
    result["Facility Type Confidence"] = result["Facility Type Original"].map(
        lambda x: lookup.get(x, {}).get("confidence", "unknown"))

    mapping_df = pd.DataFrame([
        {"Original": r, "Matched to": i["matched"],
         "Score": i["score"], "Confidence": i["confidence"]}
        for r, i in sorted(lookup.items(), key=lambda x: x[1]["score"], reverse=True)
    ])
    low_conf = mapping_df[mapping_df["Confidence"] == "low"]
    if not low_conf.empty:
        print(f"\n  ⚠️  {len(low_conf)} low-confidence matches — review:")
        print(low_conf.to_string(index=False))
    print(f"\n  Confidence breakdown:")
    print(mapping_df["Confidence"].value_counts().to_string())

    return result


# ══════════════════════════════════════════════════════════════
#  MASTER PIPELINE
# ══════════════════════════════════════════════════════════════

def clean_dataframe(raw_df: pd.DataFrame) -> pd.DataFrame:
    """
    Cleaning order and what each step drops:

    Step 0 — Violations  : normalises delimiters + code prefixes (no drops)
    Step 1 — Risk        : drops rows with invalid Risk values
    Step 2 — Address     : flags invalid addresses (no drops)
    Step 3 — Coords/Geo  : coord correction (no drops)
                           drops non-IL State
                           drops non-Chicago City (if DROP_NON_TARGET_CITIES)
                           drops invalid Zip codes
    Step 4 — Facility Type: NLP remapping (no drops)
    """
    start      = time.time()
    rows_start = len(raw_df)

    print("── Step 0: Violations ───────────────────────────────────")
    result = clean_violations(raw_df)

    print("\n── Step 1: Risk ─────────────────────────────────────────")
    result = clean_risk(result)

    print("\n── Step 2: Address ──────────────────────────────────────")
    result = clean_address_column(result)

    print("\n── Step 3: Location / State / City / Zip ────────────────")
    result = clean_location_state_city_zip(result)

    print("\n── Step 4: Facility Type NLP ────────────────────────────")
    print("  Loading model...")
    model  = SentenceTransformer("all-MiniLM-L6-v2")
    result = clean_facility_type_column(result, model)

    print(f"\n{'═'*65}")
    print(f"  PIPELINE COMPLETE  ({time.time() - start:.1f}s)")
    print(f"{'═'*65}")
    print(f"  Rows in            : {rows_start:,}")
    print(f"  Rows out           : {len(result):,}")
    print(f"  Total dropped      : {rows_start - len(result):,}")
    print(f"\n  Drop breakdown:")
    print(f"    Step 1 Risk      : invalid Risk value")
    print(f"    Step 3 State     : State != IL (after coord correction)")
    print(f"    Step 3 City      : City != CHICAGO (after fuzz match)")
    print(f"    Step 3 Zip       : invalid Chicago zip prefix/length")
    print(f"\n  No drops in:")
    print(f"    Step 0 Violations: delimiter + prefix normalisation")
    print(f"    Step 2 Address   : {(~result['address_is_valid']).sum():,} flagged in address_is_valid")
    print(f"    Step 4 Facility  : NLP remapping only")
    return result.reset_index(drop=True)


# ── Run ───────────────────────────────────────────────────────
cleaned_df = clean_dataframe(df)

── Step 0: Violations ───────────────────────────────────
  Violations normalised : 194,191 non-null entries
  Delimiter detected    : '|'

── Step 1: Risk ─────────────────────────────────────────
  Valid Risk rows   : 267,398
  Dropped (invalid) : 133
  Invalid values seen:
Risk
NAN    81
ALL    52

── Step 2: Address ──────────────────────────────────────
  Valid addresses   : 249,266
  Invalid addresses : 18,132 (flagged, not dropped)

── Step 3: Location / State / City / Zip ────────────────

  [3a] Fixing State/City from Location coordinates...
  State corrected to IL      : 59
  City corrected to CHICAGO  : 270
  Null Locations filled      : 920  (avg: 41.880734, -87.676429)

  [3b] Dropping non-IL rows...
  Dropped (non-IL) : 11

  [3c] Cleaning City...
  Non-Chicago rows : 195  (dropping)

  [3d] Cleaning Zip...
  Dropped (invalid Zip) : 54

── Step 4: Facility Type NLP ────────────────────────────
  Loading model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]


  ⚠️  229 low-confidence matches — review:
                              Original                                                       Matched to  Score Confidence
                        Shared Kitchen                                                          Caterer 0.4994        low
                      employee kitchen                                                          Caterer 0.4968        low
                       CHURCH/DAY CARE Day Care Center (combo, for ages less than 2 and 2 - 6 combined) 0.4965        low
                             CAFETERIA                                                           School 0.4963        low
                             cafeteria                                                           School 0.4963        low
                             Cafeteria                                                           School 0.4963        low
                          smoothie bar                                                      Coffee Sho